### Middleware
<br>
Middleware provides a way to more tightly control what happends inside the agent, middle is useful for following:
<br>
1. tracking agent behaviour with logging, analytics, and debugging
<br>
2. transforming prompts, tool selection, and output formatting
<br>
3. adding retries, fallbacks, and early termination logic
<br>
4. applying rate limits, guardrailes

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

<b>Summarization middleware</b>: it automatically suimmarizes conversation history when approaching token limits, preserving recent messages while compresing older context

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    checkpointer= InMemorySaver(),
    middleware=[ SummarizationMiddleware(
        model = 'groq:qwen/qwen3-32b',  #we must use  lighter cost saving model for summarization
        trigger = ("messages",10), #when message length exceeds it triggers the summarization middleware for past messages
        keep = ("messages",4) #preserve recent top 4 messages

    )]
)

just creating a config user to test the agent

In [3]:
config = {"configurable":{"thread_id":"test-1"}}

In [4]:
questions = [
    "what is 1+1?",
    "what is 1+4?",
    "what is 1+7?",
    "what is 1+10?",
    "what is 1+3?",
    "what is 1+9?",
    "what is 1+6?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content = q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 1+1?', additional_kwargs={}, response_metadata={}, id='49bc0bad-c3c0-4c2a-9f71-29618f36b242'), AIMessage(content='<think>\nOkay, the user is asking "what is 1+1?" Hmm, that seems pretty straightforward. Let me make sure I understand the question correctly. They want the sum of 1 and 1. In basic arithmetic, adding 1 and 1 gives 2. But wait, maybe they\'re looking for something more? Like in different contexts, such as binary or modulo arithmetic? Let me think.\n\nIn standard base-10 (decimal) system, 1 + 1 is indeed 2. But if we\'re talking about binary, 1 + 1 would be 10, since binary only uses 0 and 1. However, the question doesn\'t specify the numeral system, so it\'s safe to assume it\'s decimal. Also, in modulo 2 arithmetic, 1 + 1 would be 0, because any number modulo 2 cycles every two. But again, unless stated otherwise, decimal is the default.\n\nAnother angle: sometimes people might ask this as a joke or a trick question. But

Summarization on tokens

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool

@tool
def search_hotels(city:str) ->str:
  """search hotels - return long response to use more tokens"""
  return f"""Hotels in {city}:
  1. Grand hotel - 5 star,
  2. City inn - 4 star,
  3. Deluxe world - 3 star"""


agent = create_agent(
    model= 'groq:qwen/qwen3-32b',
    checkpointer= InMemorySaver(),
    tools=[search_hotels],
    middleware = [
      SummarizationMiddleware(
        model = 'groq:qwen/qwen3-32b',
        trigger = ("tokens",550),
        keep = ("tokens",200)
      )
    ]
  )

In [6]:
config = {"configurable":{"thread_id":"test-2"}}

In [7]:
def count_tokens(messages):
    totla_chars = sum(len(str(m.content))for m in messages)
    return totla_chars // 4 #4 chars = 1 token

In [9]:
cities = ['Paris', 'Mumbai', 'New york']

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config= config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")



Paris: ~1192 tokens, 8 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s tackle this step by step. The user wants me to extract the most important context from the conversation history to replace it, saving tokens. The instructions are pretty clear, but I need to make sure I don\'t miss any key points.\n\nFirst, I need to figure out the session intent. The user started by asking about hotels in Paris, got a list, and then shifted to Mumbai. The primary goal seems to be finding and booking hotels, with the latest focus on Mumbai. So the session intent is probably about identifying and booking hotels in Mumbai after an initial Paris inquiry.\n\nNext, the summary section needs the most important context from the conversation. The user requested Paris hotels first, which were provided, then moved to Mumbai. The AI used a tool to search for Mumbai hotels and gave options. The user hasn\'t confirmed a booking yet, so the next steps involv

Human in the loop middleware: pauses execution for human approval, editing or rejection of tool calls before they execute<bR>
They are usuakly used for human approval in database writesm financial transtions, etc

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipent: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipent} with subject{subject}"

In [14]:
agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                'send_email_tool':{
                    'allowed_decision':['approve','edit','reject']
                },
                'read_email_tool': False, #here no interrupt for this function
            }
        )
    ]
)